In [ ]:
import pandas as pd
from gensim.models import Word2Vec
from gensim.parsing.preprocessing import preprocess_string
from sklearn.metrics.pairwise import cosine_similarity

from pathlib import Path

dataset_root = Path("LP IV Updated Datasets")
if not dataset_root.is_dir():
    dataset_root = Path("../LP IV Updated Datasets")
print("Dataset folder:", dataset_root.resolve())


In [ ]:
# Load and inspect tweets
text = pd.read_csv(dataset_root / "CBOW/twitter_dataset.csv").Text.dropna()
print("Loaded tweets:", len(text))
print("Sample tweets:")
for i in range(3):
    print(i + 1, text.iloc[i][:200])


In [ ]:
# Clean and stem tweets
tweets = text.to_frame()
sentences = []
for tweet in text:
    words = preprocess_string(tweet)
    sentences.append(words)
tweets["words"] = sentences
tweets = tweets[tweets.words.str.len() > 0]
print("Nonempty prepared tweets:", len(tweets))
print("First tweet tokens:", tweets.words.iloc[0][:20])


In [ ]:
# Train CBOW and average word vectors for each tweet
cbow = Word2Vec(tweets.words, min_count=1, sg=0, epochs=20)
print("Vocabulary size:", len(cbow.wv))
print("Word vector size:", cbow.vector_size)
print("Sample vocabulary:", cbow.wv.index_to_key[:10])

vectors = []
for words in tweets.words:
    vector = cbow.wv.get_mean_vector(words)
    vectors.append(vector)
print("Document vector count:", len(vectors))
print("Document vector dimensions:", len(vectors[0]))
print("First document vector sample:", vectors[0][:10])


In [ ]:
# Rerun this cell to enter a new query without retraining
words = preprocess_string(input("Enter keywords: "))
print("Cleaned query tokens:", words)
matched_words = []
for word in words:
    if word in cbow.wv:
        matched_words.append(word)
print("Query tokens found in vocabulary:", matched_words[:20])
query = cbow.wv.get_mean_vector(words)
print("Query vector sample:", query[:10])

tweets["score"] = cosine_similarity([query], vectors)[0]
print("First five cosine scores:")
print(tweets["score"].head().to_string())

recommendations = tweets.nlargest(5, "score")[["Text", "score"]].copy()
recommendations["Text"] = recommendations.Text.str.replace("\n", " ", regex=False).str.slice(0, 200)
print("Top five recommendations (text shortened to 200 characters):")
print(recommendations.to_string(index=False))
